# Faster R-CNN training

In [3]:
!pip install torchmetrics
import os
import random
import time
import re
import shutil
from pathlib import Path
from collections import defaultdict, Counter
from itertools import islice
from concurrent.futures import ProcessPoolExecutor
import warnings

import cv2
import imageio.v3 as imageio
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import patches
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as func
from torchvision.ops.boxes import box_iou
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.ops import box_iou
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as TF
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import pandas as pd
import json
import ast

from tqdm import tqdm

from torch.amp import GradScaler, autocast
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FasterRCNN_ResNet50_FPN_Weights

import copy
import seaborn as sns

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 63.9 MB/s eta 0:00:00


## Setup

In [ ]:
PROCESSED_DATA_PATH = Path("../data/processed")
MODELS_PATH = Path("../models")

image_folder_path = PROCESSED_DATA_PATH / "images"

coco_json_path = PROCESSED_DATA_PATH / "coco_annotations_new.json"
new_coco_json_path = PROCESSED_DATA_PATH / "mod_coco_annotations_new.json"

train_path = PROCESSED_DATA_PATH / "train_new.json"
val_path = PROCESSED_DATA_PATH / "val_new.json"
test_path = PROCESSED_DATA_PATH / "test_new.json"

# model_path = MODELS_PATH / "FasterRCNN/ResNet50_best_model.pth"

RANDOM_SEED = 42

In [6]:
# Suppress specific warnings from the skimage module
warnings.filterwarnings("ignore",
    message="Applying `local_binary_pattern` to floating-point images may give unexpected results.*")

In [7]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Utility

In [8]:
def load_json(file_path):
    """
    Load a JSON file from the specified path.

    :param file_path: Path to the JSON file to load.
    :return: Data contained in the JSON file (as a dictionary or list).
    """
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

In [9]:
def random_filename(json_path):
    data = load_json(json_path)
    filenames = {img['file_name'] for img in data.get("images", [])}

    # Convert the set to a list to avoid the DeprecationWarning
    random_filenames = random.sample(list(filenames), min(2, len(filenames)))
    return random_filenames

In [10]:
def display_images_with_bboxes(json_file, specific_images, images_folder, mode = 2):
    """
    Visualize the specified images with all bounding boxes over them.

    :param json_file: path to the JSON file containing images, annotations, and categories
    :param specific_images: list of image names to visualize
    :param images_folder: path to the folder containing the images
    """
    # Load the JSON
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Extract images and annotations
    images = data["images"]
    annotations = data["annotations"]

    # Create a dictionary to map image IDs to file names
    image_dict = {image["id"]: image["file_name"] for image in images}

    # Filter annotations for the specific images
    specific_annotations = [ann for ann in annotations if image_dict[ann["image_id"]] in specific_images]

    # Create a dictionary to collect all annotations for each image
    image_bboxes = {}
    for annotation in specific_annotations:
        image_name = image_dict[annotation["image_id"]]
        if image_name not in image_bboxes:
            image_bboxes[image_name] = []
        if mode == 1:
            bbox = ast.literal_eval(annotation["bbox"])
        else:
            bbox = annotation["bbox"]
        category_id = annotation["category_id"]
        image_bboxes[image_name].append((bbox, category_id))

    # Visualize all images with all bounding boxes
    for image_name, bboxes in image_bboxes.items():
        # Load the image
        image_path = f'{images_folder}/{image_name}'  # Use the correct path for the images
        image = Image.open(image_path)

        # Create the figure for visualization
        plt.figure(figsize=(8, 8))
        plt.imshow(image)

        # Add all bounding boxes and category_id
        if mode == 1:
            for bbox, category_id in bboxes:
                x, y, w, h = bbox
                plt.gca().add_patch(plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none'))
                plt.text(x, y - 5, f'Category: {category_id}', color='red', fontsize=10, backgroundcolor='white')
        else:
            for bbox, category_id in bboxes:
                x, y, x2, y2 = bbox
                w = x2 - x
                h = y2 - y
                plt.gca().add_patch(plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none'))
                plt.text(x, y - 5, f'Category: {category_id}', color='red', fontsize=10, backgroundcolor='white')

        # Set the title and turn off the axes
        plt.title(f"Image: {image_name}")
        plt.axis('off')
        plt.show()

In [71]:
def display_images_with_bboxes_dataloader(dataloader, num_images=1):
    """
    Visualizza un batch di immagini con i relativi bounding box in un layout 2x2.

    :param dataloader: il DataLoader che fornisce i dati (immagini, annotazioni)
    :param num_images: numero di immagini da visualizzare
    """
    # Ottieni il primo batch di dati dal dataloader
    images, targets = next(iter(dataloader))

    # Limita il numero di immagini da visualizzare
    images = images[:num_images]
    targets = targets[:num_images]

    # Imposta la figura con 2 righe e 2 colonne
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.flatten()  # Converte l'array 2D di assi in 1D per un facile accesso

    for i in range(len(images)):
        image_tensor = images[i]
        target = targets[i]

        # Converti il tensore dell'immagine in un formato visualizzabile
        image = image_tensor.permute(1, 2, 0).numpy()  # Cambia la dimensione da (C, H, W) a (H, W, C)
        image = np.clip(image, 0, 1)  # Garantisce che i valori siano nel range [0, 1]

        # Seleziona l'asse corrispondente per la visualizzazione
        ax = axes[i]
        ax.imshow(image)

        # Aggiungi tutti i bounding box
        bboxes = target['boxes']
        labels = target['labels']

        for j in range(len(bboxes)):
            bbox = bboxes[j].numpy()
            category_id = labels[j].item()

            x, y, x2, y2 = bbox
            w = x2 - x
            h = y2 - y
            ax.add_patch(plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none'))
            ax.text(x, y - 5, f'Category: {category_id}', color='red', fontsize=10, backgroundcolor='white')

        # Imposta il titolo per ogni immagine
        ax.set_title(f"Image {i + 1}")
        ax.axis('off')

    plt.tight_layout()  # Migliora la disposizione delle sottotrame
    plt.show()

In [23]:
def extract_categories_from_coco_json(json_path):
    """
    Extracts category names from a COCO format JSON file, sorted by their ID.

    Args:
        json_path (str): Path to the COCO JSON file.

    Returns:
        list: List of category names sorted by ID.
    """
    # Read the JSON file
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Get and sort categories by ID
    categories = sorted(data.get('categories', []), key=lambda x: x['id'])

    # Extract only the category names
    category_names = [cat["name"] for cat in categories]

    return category_names

In [24]:
def plot_category_distribution(json_path):
    # Load the JSON
    with open(json_path, 'r') as file:
        data = json.load(file)

    # Extract annotations
    annotations = data['annotations']

    # Count category occurrences
    category_counts = Counter(item['category_id'] for item in annotations)

    # Prepare data for the plot
    categories = list(category_counts.keys())
    counts = list(category_counts.values())

    # Create the plot
    plt.figure(figsize=(10, 6))
    plt.bar(categories, counts, color='skyblue')
    plt.xlabel('Category ID')
    plt.ylabel('Frequency')
    plt.title('Category Distribution in the Dataset')
    plt.xticks(categories)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

## Dataset preprocessing

In [25]:
def process_custom_coco_json(input_path, output_path):
    """
    Function to process a custom COCO format JSON.
    """
    # Read the JSON from the input file
    data = load_json(input_path)

    # Get and correct the format of the categories
    raw_categories = data.get('categories', [])
    categories = []

    for category in tqdm(raw_categories, desc="Processing Categories"):
        for id_str, name in category.items():
            try:
                categories.append({"id": int(id_str), "name": name})
            except ValueError:
                print(f"Error parsing category: {category}")

    # Find the "Aircraft" category with ID 0
    aircraft_category = next((cat for cat in categories if cat['id'] == 0 and cat['name'] == "Aircraft"), None)
    if aircraft_category:
        aircraft_category['id'] = 11  # Change the ID of the "Aircraft" category to 11

    # Add the "background" category with ID 0 if it doesn't exist
    if not any(cat['id'] == 0 for cat in categories):
        categories.append({"id": 0, "name": "background"})

    # Preprocess the annotations into a dictionary by image
    image_annotations_dict = {}
    for annotation in tqdm(data.get('annotations', []), desc="Building Image Annotations Dictionary"):
        image_id = annotation['image_id']
        if image_id not in image_annotations_dict:
            image_annotations_dict[image_id] = []
        image_annotations_dict[image_id].append(annotation)

    # List of annotations to keep (only valid ones)
    valid_annotations = []
    annotations_to_remove = set()

    # Check bounding boxes
    for annotation in tqdm(data.get('annotations', []), desc="Processing Annotations"):
        if annotation['category_id'] == 0:  # If it's Aircraft
            annotation['category_id'] = 11

        # Convert the bbox format
        if isinstance(annotation['bbox'], str):
            annotation['bbox'] = json.loads(annotation['bbox'])

        x, y, width, height = annotation['bbox']
        xmin, xmax = x, x + width
        ymin, ymax = y, y + height

        # Check that xmin < xmax and ymin < ymax, and that the width and height are sufficient
        if xmin >= xmax or ymin >= ymax or width <= 10 or height <= 10:
            annotations_to_remove.add(annotation['id'])
        else:
            annotation['bbox'] = [xmin, ymin, xmax, ymax]
            valid_annotations.append(annotation)

    # Remove invalid annotations
    data['annotations'] = valid_annotations

    # Update the categories in the JSON
    data['categories'] = categories

    # Randomly select 10% of the images
    all_images = data.get('images', [])
    selected_images = random.sample(all_images, len(all_images) // 10)
    selected_image_ids = {img['id'] for img in selected_images}

    # Filter images and annotations
    data['images'] = selected_images
    data['annotations'] = [ann for ann in data['annotations'] if ann['image_id'] in selected_image_ids]

    # Write the modified JSON to the output file
    with open(output_path, 'w') as f:
        json.dump(data, f, indent=4)

In [26]:
process_custom_coco_json(coco_json_path, new_coco_json_path)

Processing Annotations: 100%|██████████| 669984/669984 [00:03<00:00, 188522.50it/s]


In [27]:
old_json = load_json(coco_json_path)

In [28]:
# Count the number of images, annotations, and categories
num_images = len(old_json.get("images", []))
num_annotations = len(old_json.get("annotations", []))
num_categories = len(old_json.get("categories", []))

print(f"Number of images: {num_images}")
print(f"Number of annotations: {num_annotations}")
print(f"Number of categories: {num_categories}")

Number of images: 45891
Number of annotations: 669984
Number of categories: 11


In [29]:
new_json = load_json(new_coco_json_path)

In [30]:
# Count the number of images, annotations, and categories
num_images = len(new_json.get("images", []))
num_annotations = len(new_json.get("annotations", []))
num_categories = len(new_json.get("categories", []))

print(f"Number of images: {num_images}")
print(f"Number of annotations: {num_annotations}")
print(f"Number of categories: {num_categories}")

Number of images: 4589
Number of annotations: 47406
Number of categories: 12


In [31]:
def get_common_random_filenames(json_file1, json_file2):
    # Load the JSON files
    with open(json_file1, 'r') as f1, open(json_file2, 'r') as f2:
        data1 = json.load(f1)
        data2 = json.load(f2)

    # Extract the file names from both JSONs
    filenames1 = {img['file_name'] for img in data1.get("images", [])}
    filenames2 = {img['file_name'] for img in data2.get("images", [])}

    # Find the common file names
    common_filenames = list(filenames1 & filenames2)

    random_filenames = random.sample(common_filenames, min(2, len(common_filenames)))

    return random_filenames

In [32]:
common_random_files = get_common_random_filenames(coco_json_path, new_coco_json_path)
print("OLD JSON")
display_images_with_bboxes(coco_json_path, common_random_files, image_folder_path, mode = 1)
print("NEW JSON")
display_images_with_bboxes(new_coco_json_path, common_random_files, image_folder_path)

OLD JSON
NEW JSON


## Undersampling and upsampling

In [33]:
def count_bboxes_per_category(json_path):
    """
    Function that counts the number of bounding boxes for each category in a COCO format JSON file.

    :param json_path: Path to the JSON file.
    :return: Dictionary with category names as keys and bounding box counts as values.
    """
    # Read the JSON file
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Get category mapping (id -> name)
    category_mapping = {cat['id']: cat['name'] for cat in data.get('categories', [])}

    # Count bounding boxes for each category_id
    bbox_counts = defaultdict(int)
    for annotation in data.get('annotations', []):
        category_id = annotation['category_id']
        bbox_counts[category_id] += 1

    # Convert the counts to use category names
    bbox_counts_named = {category_mapping[cat_id]: count for cat_id, count in bbox_counts.items()}

    return bbox_counts_named

In [34]:
bbox_counts = count_bboxes_per_category(new_coco_json_path)

# Print the results
for category, count in bbox_counts.items():
    print(f"Category: {category}, Number of bboxes: {count}")

Category: Passenger Vehicle, Number of bboxes: 9577
Category: Building, Number of bboxes: 33202
Category: Truck, Number of bboxes: 2423
Category: Engineering Vehicle, Number of bboxes: 420
Category: Railway Vehicle, Number of bboxes: 324
Category: Aircraft, Number of bboxes: 137
Category: Shipping Container, Number of bboxes: 536
Category: Maritime Vessel, Number of bboxes: 570
Category: Storage Tank, Number of bboxes: 166
Category: Helipad, Number of bboxes: 14
Category: Pylon, Number of bboxes: 37


In [ ]:
plot_category_distribution(new_coco_json_path)

In [36]:
def undersample_class(json_data_path, dataset_out, target_class=6, target_percentage=0.9):
    # Load the JSON file
    def load_json(file_path):
        with open(file_path, "r") as file:
            return json.load(file)

    json_data = load_json(json_data_path)

    # Count the occurrences of each class per image
    image_class_counts = Counter()
    total_class_counts = Counter()
    for annotation in json_data["annotations"]:
        image_class_counts[(annotation["image_id"], annotation["category_id"])] += 1
        total_class_counts[annotation["image_id"]] += 1

    # Identify images with at least 90% of annotations in the target class
    images_with_majority_target = {
        image_id for (image_id, category_id), count in image_class_counts.items()
        if category_id == target_class and count / total_class_counts[image_id] >= target_percentage
    }

    # Remove images that contain only the target class
    images_with_only_target = {
        image_id for (image_id, category_id), count in image_class_counts.items()
        if category_id == target_class and total_class_counts[image_id] == count
    }

    # Union of the two conditions: only target or majority target
    images_to_remove = images_with_majority_target | images_with_only_target

    # Filter images and annotations to remove unnecessary ones
    remaining_images = [
        img for img in json_data["images"]
        if img["id"] not in images_to_remove
    ]
    remaining_annotations = [
        ann for ann in json_data["annotations"]
        if ann["image_id"] not in images_to_remove
    ]

    # Update the JSON
    json_data["images"] = remaining_images
    json_data["annotations"] = remaining_annotations

    # Save the result to a new file
    with open(dataset_out, "w") as file:
        json.dump(json_data, file, indent=4)

    print(f"Images processed and undersampling completed.")

In [37]:
undersample_class(new_coco_json_path, new_coco_json_path, target_class=6)

Images processed and undersampling completed.


In [38]:
new_json = load_json(new_coco_json_path)

# Count the number of images, annotations, and categories
num_images = len(new_json.get("images", []))
num_annotations = len(new_json.get("annotations", []))
num_categories = len(new_json.get("categories", []))

print(f"Number of images: {num_images}")
print(f"Number of annotations: {num_annotations}")
print(f"Number of categories: {num_categories}")

Number of images: 3326
Number of annotations: 25706
Number of categories: 12


In [39]:
bbox_counts = count_bboxes_per_category(new_coco_json_path)

# Print the results
for category, count in bbox_counts.items():
    print(f"Category: {category}, Number of bboxes: {count}")

Category: Passenger Vehicle, Number of bboxes: 9188
Category: Building, Number of bboxes: 12004
Category: Truck, Number of bboxes: 2331
Category: Engineering Vehicle, Number of bboxes: 413
Category: Railway Vehicle, Number of bboxes: 324
Category: Aircraft, Number of bboxes: 135
Category: Shipping Container, Number of bboxes: 531
Category: Maritime Vessel, Number of bboxes: 566
Category: Storage Tank, Number of bboxes: 163
Category: Helipad, Number of bboxes: 14
Category: Pylon, Number of bboxes: 37


In [ ]:
plot_category_distribution(new_coco_json_path)

In [41]:
def upsample_classes(json_data_path, dataset_out, classes_to_upsample, value):
    json_data = load_json(json_data_path)

    # Count the current occurrences for each class
    class_bbox_counts = Counter()
    image_annotations = {}

    for annotation in json_data["annotations"]:
        if annotation["category_id"] in classes_to_upsample:
            class_bbox_counts[annotation["category_id"]] += 1
        if annotation["image_id"] not in image_annotations:
            image_annotations[annotation["image_id"]] = []
        image_annotations[annotation["image_id"]].append(annotation)

    # Identify images with priority for each class to upsample
    class_images = {class_id: [] for class_id in classes_to_upsample}
    for img in json_data["images"]:
        image_id = img["id"]
        annotations = image_annotations.get(image_id, [])
        target_annotations = [ann for ann in annotations if ann["category_id"] in classes_to_upsample]

        if target_annotations:
            # Rank images based on the number of target class bboxes
            for ann in target_annotations:
                class_images[ann["category_id"]].append((img, annotations))

    # Sort images for each class based on the number of bboxes
    for class_id in classes_to_upsample:
        class_images[class_id].sort(key=lambda x: len([ann for ann in x[1] if ann["category_id"] == class_id]), reverse=True)

    # Create new images and annotations to increase occurrences
    new_images = []
    new_annotations = []
    image_id_offset = max(img["id"] for img in json_data["images"]) + 1
    annotation_id_offset = max(ann["id"] for ann in json_data["annotations"]) + 1

    for class_id in classes_to_upsample:
        class_data = class_images[class_id]
        idx = 0
        current_count = class_bbox_counts[class_id]
        target_count = value * class_bbox_counts[class_id]

        while current_count < target_count:
            img, annotations = class_data[idx % len(class_data)]

            # Duplicate the image
            new_img = copy.deepcopy(img)
            new_img["id"] = image_id_offset

            # Duplicate the annotations associated with the target class
            for ann in annotations:
                if ann["category_id"] == class_id:
                    new_ann = copy.deepcopy(ann)
                    new_ann["id"] = annotation_id_offset
                    new_ann["image_id"] = image_id_offset
                    new_annotations.append(new_ann)
                    annotation_id_offset += 1

            new_images.append(new_img)

            # Increment the offsets and the count
            image_id_offset += 1
            current_count += len([ann for ann in annotations if ann["category_id"] == class_id])
            idx += 1

    # Add the new images and annotations to the JSON
    json_data["images"].extend(new_images)
    json_data["annotations"].extend(new_annotations)

    # Save the result to a new file
    with open(dataset_out, "w") as file:
        json.dump(json_data, file, indent=4)

    print(f"Upsampling completed.")

In [42]:
# List of classes to upsample and the target number of occurrences for each
classes_to_upsample = [2, 3, 4, 5, 7, 8, 9, 10, 11]

# Call the function
upsample_classes(new_coco_json_path, new_coco_json_path, classes_to_upsample, value = 3)

Upsampling completed.


In [43]:
new_json = load_json(new_coco_json_path)

# Count the number of images, annotations, and categories
num_images = len(new_json.get("images", []))
num_annotations = len(new_json.get("annotations", []))
num_categories = len(new_json.get("categories", []))

print(f"Number of images: {num_images}")
print(f"Number of annotations: {num_annotations}")
print(f"Number of categories: {num_categories}")

Number of images: 3656
Number of annotations: 34784
Number of categories: 12


In [44]:
# List of classes to upsample and the target number of occurrences for each
classes_to_upsample = [7, 8, 10, 11]

# Call the function
upsample_classes(new_coco_json_path, new_coco_json_path, classes_to_upsample, value = 10)

Upsampling completed.


In [45]:
new_json = load_json(new_coco_json_path)

# Count the number of images, annotations, and categories
num_images = len(new_json.get("images", []))
num_annotations = len(new_json.get("annotations", []))
num_categories = len(new_json.get("categories", []))

print(f"Number of images: {num_images}")
print(f"Number of annotations: {num_annotations}")
print(f"Number of categories: {num_categories}")

Number of images: 5403
Number of annotations: 44309
Number of categories: 12


In [46]:
# List of classes to upsample and the target number of occurrences for each
classes_to_upsample = [1]

# Call the function
upsample_classes(new_coco_json_path, new_coco_json_path, classes_to_upsample, value = 1.2)

Upsampling completed.


In [47]:
new_json = load_json(new_coco_json_path)

# Count the number of images, annotations, and categories
num_images = len(new_json.get("images", []))
num_annotations = len(new_json.get("annotations", []))
num_categories = len(new_json.get("categories", []))

print(f"Number of images: {num_images}")
print(f"Number of annotations: {num_annotations}")
print(f"Number of categories: {num_categories}")

Number of images: 5410
Number of annotations: 46311
Number of categories: 12


In [48]:
bbox_counts = count_bboxes_per_category(new_coco_json_path)

# Print the results
for category, count in bbox_counts.items():
    print(f"Category: {category}, Number of bboxes: {count}")

Category: Passenger Vehicle, Number of bboxes: 11190
Category: Building, Number of bboxes: 12004
Category: Truck, Number of bboxes: 7007
Category: Engineering Vehicle, Number of bboxes: 1239
Category: Railway Vehicle, Number of bboxes: 972
Category: Aircraft, Number of bboxes: 4070
Category: Shipping Container, Number of bboxes: 1602
Category: Maritime Vessel, Number of bboxes: 1716
Category: Storage Tank, Number of bboxes: 4980
Category: Helipad, Number of bboxes: 420
Category: Pylon, Number of bboxes: 1111


In [ ]:
plot_category_distribution(new_coco_json_path)

In [ ]:
print("FINAL JSON")
display_images_with_bboxes(new_coco_json_path, random_filename(new_coco_json_path), image_folder_path)

## Splitting

In [51]:
def split(json_file, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    # Load the JSON
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Filter annotations with invalid bounding boxes
    valid_annotations = data['annotations']

    # Get the list of images
    images = data['images']

    # Shuffle the image IDs randomly
    random.shuffle(images)

    # Calculate the limits for train, validation, and test
    total_images = len(images)
    total_annotations = len(valid_annotations)
    train_end = int(total_images * train_ratio)
    val_end = int(total_images * (train_ratio + val_ratio))

    # Split the images into respective sets
    train_images = images[:train_end]
    val_images = images[train_end:val_end]
    test_images = images[val_end:]

    # Group image IDs for respective sets
    train_image_ids = {image['id'] for image in train_images}
    val_image_ids = {image['id'] for image in val_images}
    test_image_ids = {image['id'] for image in test_images}

    # Filter annotations for respective image sets
    train_annotations = [ann for ann in valid_annotations if ann['image_id'] in train_image_ids]
    val_annotations = [ann for ann in valid_annotations if ann['image_id'] in val_image_ids]
    test_annotations = [ann for ann in valid_annotations if ann['image_id'] in test_image_ids]

    # Create new JSONs for train, validation, and test
    train_data = {'images': train_images, 'annotations': train_annotations, 'categories': data['categories']}
    val_data = {'images': val_images, 'annotations': val_annotations, 'categories': data['categories']}
    test_data = {'images': test_images, 'annotations': test_annotations, 'categories': data['categories']}

    # Save the JSON files
    with open('train.json', 'w') as f:
        json.dump(train_data, f, indent=4)

    with open('val.json', 'w') as f:
        json.dump(val_data, f, indent=4)

    with open('test.json', 'w') as f:
        json.dump(test_data, f, indent=4)

    # Check the proportion of images and annotations
    check_split_proportions(total_images, total_annotations,
                            len(train_images), len(val_images), len(test_images),
                            len(train_annotations), len(val_annotations), len(test_annotations),
                            train_ratio, val_ratio, test_ratio,
                            train_annotations, val_annotations, test_annotations, data['categories'])


def check_split_proportions(total_images, total_annotations, train_count, val_count, test_count,
                            train_bbox_count, val_bbox_count, test_bbox_count,
                            train_ratio, val_ratio, test_ratio,
                            train_annotations, val_annotations, test_annotations, categories):
    # Percentage for images
    train_image_percentage = (train_count / total_images) * 100
    val_image_percentage = (val_count / total_images) * 100
    test_image_percentage = (test_count / total_images) * 100

    # Percentage for bbox
    train_bbox_percentage = (train_bbox_count / total_annotations) * 100
    val_bbox_percentage = (val_bbox_count / total_annotations) * 100
    test_bbox_percentage = (test_bbox_count / total_annotations) * 100

    print(f"Total images: {total_images}")
    print(f"Total annotations (bbox): {total_annotations}")
    print(f"Train: {train_count} images ({train_image_percentage:.2f}%) ({train_bbox_count} bbox) ({train_bbox_percentage:.2f}%)")
    print(f"Val: {val_count} images ({val_image_percentage:.2f}%) ({val_bbox_count} bbox) ({val_bbox_percentage:.2f}%)")
    print(f"Test: {test_count} images ({test_image_percentage:.2f}%) ({test_bbox_count} bbox) ({test_bbox_percentage:.2f}%)")

    # Calculate the number of annotations per category in each set
    category_count_train = defaultdict(int)
    category_count_val = defaultdict(int)
    category_count_test = defaultdict(int)

    for annotation in train_annotations:
        category_count_train[annotation['category_id']] += 1
    for annotation in val_annotations:
        category_count_val[annotation['category_id']] += 1
    for annotation in test_annotations:
        category_count_test[annotation['category_id']] += 1

    # Print the proportions per category
    print("\nProportions per category:")
    for category in categories:
        try:
            category_id = int(category['id'])  # Convert 'id' to an integer
            category_name = category['name']

            # Count the number of annotations per category in each set
            train_cat_count = category_count_train.get(category_id, 0)
            val_cat_count = category_count_val.get(category_id, 0)
            test_cat_count = category_count_test.get(category_id, 0)

            # Calculate the percentage of annotations per category
            total_cat_annotations = train_cat_count + val_cat_count + test_cat_count
            if total_cat_annotations > 0:
                train_cat_percentage = (train_cat_count / total_cat_annotations) * 100
                val_cat_percentage = (val_cat_count / total_cat_annotations) * 100
                test_cat_percentage = (test_cat_count / total_cat_annotations) * 100
            else:
                train_cat_percentage = val_cat_percentage = test_cat_percentage = 0.0

            print(f"{category_name}:")
            print(f"  Train: {train_cat_count} annotations ({train_cat_percentage:.2f}%)")
            print(f"  Val: {val_cat_count} annotations ({val_cat_percentage:.2f}%)")
            print(f"  Test: {test_cat_count} annotations ({test_cat_percentage:.2f}%)")
        except KeyError as e:
            print(f"Missing key in category: {e}")
        except ValueError as e:
            print(f"Error parsing category: {e}")


In [52]:
# Call the function
split(new_coco_json_path)

Total images: 5410
Total annotations (bbox): 46311
Train: 4328 images (80.00%) (37186 bbox) (80.30%)
Val: 541 images (10.00%) (4522 bbox) (9.76%)
Test: 541 images (10.00%) (4603 bbox) (9.94%)

Proportions per category:
Aircraft:
  Train: 3302 annotations (81.13%)
  Val: 407 annotations (10.00%)
  Test: 361 annotations (8.87%)
Passenger Vehicle:
  Train: 8894 annotations (79.48%)
  Val: 957 annotations (8.55%)
  Test: 1339 annotations (11.97%)
Truck:
  Train: 5650 annotations (80.63%)
  Val: 643 annotations (9.18%)
  Test: 714 annotations (10.19%)
Railway Vehicle:
  Train: 737 annotations (75.82%)
  Val: 141 annotations (14.51%)
  Test: 94 annotations (9.67%)
Maritime Vessel:
  Train: 1483 annotations (86.42%)
  Val: 178 annotations (10.37%)
  Test: 55 annotations (3.21%)
Engineering Vehicle:
  Train: 1012 annotations (81.68%)
  Val: 145 annotations (11.70%)
  Test: 82 annotations (6.62%)
Building:
  Train: 9686 annotations (80.69%)
  Val: 1151 annotations (9.59%)
  Test: 1167 annotatio

In [ ]:
print("TRAIN")
train_filenames = random_filename(train_path)
display_images_with_bboxes(train_path, train_filenames, image_folder_path)

In [54]:
print("VALIDATION")
val_filenames = random_filename(val_path)
display_images_with_bboxes(val_path, val_filenames, image_folder_path)

VALIDATION


In [ ]:
print("TEST")
test_filenames = random_filename(test_path)
display_images_with_bboxes(test_path, test_filenames, image_folder_path)

## Dataloader

In [55]:
class CustomDataset(Dataset):
    def __init__(self, json_file, img_dir, aug=False):
        """
        Initializes the custom dataset.
        Args:
        - json_file: The preprocessed JSON file containing images, annotations, and categories.
        - img_dir: The directory containing the images.
        - aug: Boolean to enable or disable augmentation.
        """
        # Load the preprocessed JSON file
        with open(json_file, 'r') as f:
            coco_data = json.load(f)

        # Extract information about images, annotations, and categories
        self.image_info = {image['id']: image['file_name'] for image in coco_data['images']}
        self.image_annotations = {}
        self.image_bboxes = {}

        # Extract classes (categories) from the JSON file
        self.classes = {}
        for category in coco_data['categories']:
            category_id = category['id']  # Numeric ID
            category_name = category['name']  # Category name
            self.classes[int(category_id)] = category_name


        for annotation in coco_data['annotations']:
            image_id = annotation['image_id']
            bbox = annotation['bbox']

            # Associate annotations and bounding boxes with images
            if image_id not in self.image_annotations:
                self.image_annotations[image_id] = []
                self.image_bboxes[image_id] = []

            self.image_annotations[image_id].append(annotation['category_id'])
            self.image_bboxes[image_id].append(bbox)

        # Set up image paths and select only valid images
        self.img_dir = img_dir
        self.image_paths = []
        self.image_ids = []
        for image_id, file_name in self.image_info.items():
            if image_id in self.image_annotations:
                img_path = os.path.join(img_dir, file_name)
                if os.path.exists(img_path):
                    self.image_paths.append(img_path)
                    self.image_ids.append(image_id)

        # Define transformations
        self.base_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

        self.aug_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

        self.aug = aug

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        """
        Retrieves an image and its annotations.
        """
        img_path = self.image_paths[index]
        img_id = self.image_ids[index]

        # Load the image
        image = Image.open(img_path).convert('RGB')
        original_width, original_height = image.size

        # Apply transformations
        if self.aug:
            image_tensor = self.aug_transform(image)
        else:
            image_tensor = self.base_transform(image)

        # Retrieve annotations and bounding boxes
        categories = self.image_annotations[img_id]
        bboxes = self.image_bboxes[img_id]

        # Calculate scaling
        scale_x = 224 / original_width
        scale_y = 224 / original_height

        # Resize bounding boxes
        scaled_bboxes = [
            torch.tensor([
                bbox[0] * scale_x,  # x_min
                bbox[1] * scale_y,  # y_min
                bbox[2] * scale_x,  # x_max
                bbox[3] * scale_y   # y_max
            ], dtype=torch.float32)
            for bbox in bboxes
        ]

        # Construct the target
        target = {
            "boxes": torch.stack(scaled_bboxes),
            "labels": torch.tensor(categories, dtype=torch.int64)
        }

        return image_tensor, target

In [56]:
def collate_fn(batch):
    """
    Collation function for the DataLoader, useful for batching images and annotations.
    The function will return a batch of images and a batch of targets, formatted correctly for Faster R-CNN.

    Args:
    - batch: list of tuples (image, target)

    Returns:
    - images: batch of images
    - targets: list of dictionaries containing annotations for each image
    """
    # Separate images and targets
    images, targets = zip(*batch)

    # Convert the list of images into a batch of images
    images = list(images)

    # Return the batch
    return images, list(targets)

In [57]:
train_dataset = CustomDataset(train_path, img_dir=image_folder_path, aug=True)
valid_dataset = CustomDataset(val_path, img_dir=image_folder_path, aug=False)
test_dataset = CustomDataset(test_path, img_dir=image_folder_path, aug=False)

# Creation of DataLoaders
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(valid_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

## Check dataloader

In [58]:
def validate_dataloader(dataloader):
    """
    Validates a DataLoader by checking that each image has an associated target
    and that no target is `None` or empty.

    Args:
    - dataloader: the DataLoader to validate.

    Returns:
    - error_messages: List of error messages. Empty if all data is valid.
    """
    error_messages = []
    for batch_idx, (images, targets) in enumerate(dataloader):
        for idx, target in enumerate(targets):
            if target is None:
                error_messages.append(f"Batch {batch_idx}, Image {idx}: Target is None.")
            elif target["boxes"].numel() == 0 or target["labels"].numel() == 0:
                error_messages.append(
                    f"Batch {batch_idx}, Image {idx}: Target is empty or missing 'boxes'/'labels'."
                )
    return error_messages

In [60]:
from tqdm import tqdm

def validate_dataloader(dataloader):
    error_messages = []
    for batch_idx, (images, targets) in enumerate(tqdm(dataloader, desc="Validation DataLoader")):
        for idx, target in enumerate(targets):
            if target is None:
                error_messages.append(f"Batch {batch_idx}, Image {idx}: Target is None.")
            elif target["boxes"].numel() == 0 or target["labels"].numel() == 0:
                error_messages.append(
                    f"Batch {batch_idx}, Image {idx}: Target is empty or missing 'boxes'/'labels'."
                )
    return error_messages

# Validation of the training DataLoader avec barre de progression
train_errors = validate_dataloader(train_loader)

if train_errors:
    print("Errors in the training DataLoader:")
    for error in train_errors:
        print(error)
else:
    print("All targets in the training DataLoader are valid.")

Validation DataLoader: 100%|██████████| 3132/3132 [08:22<00:00,  6.24it/s]

All targets in the training DataLoader are valid.


In [61]:
# Validation of the validation DataLoader
val_errors = validate_dataloader(val_loader)

if val_errors:
    print("Errors in the validation DataLoader:")
    for error in val_errors:
        print(error)
else:
    print("All targets in the validation DataLoader are valid.")

Validation DataLoader: 100%|██████████| 399/399 [01:34<00:00,  4.22it/s]

All targets in the validation DataLoader are valid.


In [62]:
# Validation of the test DataLoader
test_errors = validate_dataloader(test_loader)

if test_errors:
    print("Errors in the test DataLoader:")
    for error in test_errors:
        print(error)
else:
    print("All targets in the test DataLoader are valid.")

Validation DataLoader: 100%|██████████| 384/384 [01:24<00:00,  4.53it/s]

All targets in the test DataLoader are valid.


In [63]:
def count_images_and_targets(dataloader):
    """
    Count the number of images and targers in the dataloader
    """
    num_images = 0
    num_targets = 0

    for images, targets in dataloader:
        # Count the images in the batch
        num_images += len(images)

        # Count the targets for each image (number of objects)
        for target in targets:
            num_targets += len(target["boxes"])

    return num_images, num_targets

In [64]:
num_images_train, num_targets_train = count_images_and_targets(train_loader)

print(f"Total number of images in train: {num_images_train}")
print(f"Total number of targets in train: {num_targets_train}")

Total number of images in train: 3132
Total number of targets in train: 36522


In [65]:
num_images_val, num_targets_val = count_images_and_targets(val_loader)

print(f"Total number of images in validation: {num_images_val}")
print(f"Total number of targets in validation: {num_targets_val}")

Total number of images in validation: 399
Total number of targets in validation: 5323


In [66]:
num_images_test, num_targets_test = count_images_and_targets(test_loader)

print(f"Total number of images in test: {num_images_test}")
print(f"Total number of targets in test: {num_targets_test}")

Total number of images in test: 384
Total number of targets in test: 3933


In [67]:
print(f"Total number of images: {num_images_train + num_images_val +num_images_test}")
print(f"Total number of targets: {num_targets_train + num_targets_val +num_targets_test}")

Total number of images: 3915
Total number of targets: 45778


In [ ]:
print("TRAIN")
display_images_with_bboxes_dataloader(train_loader)
print("VALIDATION")
display_images_with_bboxes_dataloader(val_loader)
print("TEST")
display_images_with_bboxes_dataloader(test_loader)

## Training

In [ ]:
def train_and_validate(model, train_loader, val_loader, optimizer, device, num_epochs=10, num_classes=12, accumulation_steps=4):
    """
    Train and validate the Faster R-CNN model.
    """
    # Scaler for mixed precision training
    scaler = GradScaler()

    model.to(device)

    train_losses = []
    val_losses = []
    train_confidences = []
    val_confidences = []
    train_precisions = []
    train_recalls = []
    val_precisions = []
    val_recalls = []

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        # --------------------
        # Training
        # --------------------
        model.train()
        total_train_loss = 0.0
        train_loop = tqdm(train_loader, desc="Training", leave=False)

        for i, (images, targets) in enumerate(train_loop):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            with autocast('cuda'):
                # Calculate the loss
                loss_dict = model(images, targets)

                # Verify if the result is a dictionary
                if not isinstance(loss_dict, dict):
                    raise ValueError(f"The model gave a non valid object: {type(loss_dict)}")

                # Sum the loss
                losses = sum(loss for loss in loss_dict.values())

            # Backward pass with gradient scaling
            scaler.scale(losses).backward()

            # Gradient accumulation
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            total_train_loss += losses.item()
            train_loop.set_postfix(loss=losses.item())

        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        print(f"Average train loss: {avg_train_loss:.4f}")

        # Calculate metrics for training
        train_precision, train_recall, train_conf = evaluate_metrics(model, train_loader, device, num_classes)
        train_precisions.append(train_precision)
        train_recalls.append(train_recall)
        train_confidences.append(train_conf)
        print(f"Training precision: {train_precision:.4f}, Training recall: {train_recall:.4f}, Average training confidence: {train_conf:.4f}")

        # --------------------
        # Validation
        # --------------------
        model.train()  # Necessary to calculate the loss
        total_val_loss = 0.0
        val_loop = tqdm(val_loader, desc="Validation", leave=False)

        for i, (images, targets) in enumerate(val_loop):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Calculate the validation loss
            with torch.no_grad(), autocast('cuda'):
                loss_dict = model(images, targets)
                if not isinstance(loss_dict, dict):
                    raise ValueError(f"The model gave a non valid object: {type(loss_dict)}")
                losses = sum(loss for loss in loss_dict.values())
                total_val_loss += losses.item()
                val_loop.set_postfix(loss=losses.item())

        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        print(f"Average validation loss: {avg_val_loss:.4f}")

        # Calculate metrics for validation
        val_precision, val_recall, val_conf = evaluate_metrics(model, val_loader, device, num_classes)
        val_precisions.append(val_precision)
        val_recalls.append(val_recall)
        val_confidences.append(val_conf)
        print(f"Validation precision: {val_precision:.4f}, Validation recall: {val_recall:.4f}, Average validation confidence: {val_conf:.4f}")

        # --------------------
        # Save the best model
        # --------------------
        if epoch == 0 or avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f"ResNet50_best_model.pth")
            print(f"Best model saved: ResNet50_best_model.pth")

    return train_losses, val_losses, train_precisions, train_recalls, train_confidences, val_precisions, val_recalls, val_confidences

In [74]:
def evaluate_metrics(model, data_loader, device, num_classes, iou_threshold=0.5, score_threshold=0.2):
    """
    Calculate Precision, Recall and avergae confidence based on IoU for Faster R-CNN.
    """
    model.eval()
    true_positives = [0] * num_classes
    false_positives = [0] * num_classes
    false_negatives = [0] * num_classes
    total_confidence = 0.0
    num_valid_predictions = 0

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Evaluating", leave=False):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Predictions
            outputs = model(images)

            for target, output in zip(targets, outputs):
                target_boxes = target['boxes'].cpu()
                target_labels = target['labels'].cpu()
                pred_boxes = output['boxes'].cpu()
                pred_labels = output['labels'].cpu()
                pred_scores = output['scores'].cpu()

                # Filter predictions
                high_score_indices = pred_scores > score_threshold
                pred_boxes = pred_boxes[high_score_indices]
                pred_labels = pred_labels[high_score_indices]
                pred_scores = pred_scores[high_score_indices]

                if len(pred_boxes) > 0 and len(target_boxes) > 0:
                    ious = box_iou(pred_boxes, target_boxes)  # IoU
                    for pred_idx, pred_label in enumerate(pred_labels):
                        iou_values = ious[pred_idx]
                        max_iou_idx = torch.argmax(iou_values)
                        max_iou = iou_values[max_iou_idx]

                        if max_iou >= iou_threshold and pred_label == target_labels[max_iou_idx]:
                            true_positives[pred_label] += 1
                            total_confidence += pred_scores[pred_idx].item()
                            num_valid_predictions += 1
                            ious[:, max_iou_idx] = 0
                        else:
                            false_positives[pred_label] += 1

                # False negatives
                for target_label in target_labels:
                    if target_label not in pred_labels:
                        false_negatives[target_label] += 1

    # Global Precision and Recall
    precision = sum(true_positives) / (sum(true_positives) + sum(false_positives) + 1e-6)
    recall = sum(true_positives) / (sum(true_positives) + sum(false_negatives) + 1e-6)

    # Average confidence
    average_confidence = total_confidence / num_valid_predictions if num_valid_predictions > 0 else 0.0

    return precision, recall, average_confidence

In [75]:
def plot_metrics(train_losses, val_losses, train_precisions, train_recalls, train_confidences,
                 val_precisions, val_recalls, val_confidences, num_epochs):
    """
    Function to plot the metrics of training, validation (Loss, Precision, Recall, Confidence).
    """
    epochs_range = range(1, num_epochs + 1)

    plt.figure(figsize=(14, 12))

    # Plot Loss
    plt.subplot(3, 2, 1)
    plt.plot(epochs_range, train_losses, label='Training Loss', color='blue')
    plt.plot(epochs_range, val_losses, label='Validation Loss', color='red')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss per Epoch')

    # Plot Precision
    plt.subplot(3, 2, 2)
    plt.plot(epochs_range, train_precisions, label='Training Precision', color='green')
    plt.plot(epochs_range, val_precisions, label='Validation Precision', color='orange')
    plt.xlabel('Epochs')
    plt.ylabel('Precision')
    plt.legend()
    plt.title('Precision per Epoch')

    # Plot Recall
    plt.subplot(3, 2, 3)
    plt.plot(epochs_range, train_recalls, label='Training Recall', color='purple')
    plt.plot(epochs_range, val_recalls, label='Validation Recall', color='pink')
    plt.xlabel('Epochs')
    plt.ylabel('Recall')
    plt.legend()
    plt.title('Recall per Epoch')

    # Plot Confidence
    plt.subplot(3, 2, 4)
    plt.plot(epochs_range, train_confidences, label='Training Confidence', color='cyan')
    plt.plot(epochs_range, val_confidences, label='Validation Confidence', color='magenta')
    plt.xlabel('Epochs')
    plt.ylabel('Confidence')
    plt.legend()
    plt.title('Confidence per Epoch')

    plt.tight_layout()
    plt.show()

In [ ]:
# Load the Faster R-CNN model with ResNet50 and FPN
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 175MB/s]


In [77]:
num_classes = 12

# Modify the number of classes in output
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

# Freeze the backbone layer (ResNet50)
for param in model.backbone.parameters():
    param.requires_grad = True

# Device = GPU or CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Configure training
num_epochs = 10
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

model.to(device)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [78]:
train_losses, val_losses, train_precisions, train_recalls, train_confidences, val_precisions, val_recalls, val_confidences = train_and_validate(model, train_loader, val_loader, optimizer, device, num_epochs)


Epoch 1/10


Average train loss: 0.7751


Training precision: 0.3346, Training recall: 0.9605, Average training confidence: 0.7547


Average validation loss: 0.6447


Validation precision: 0.3557, Validation recall: 0.9546, Average validation confidence: 0.7377
Model saved: model_epoch_1.pth

Epoch 2/10


...

Validation precision: 0.6192, Validation recall: 0.9530, Average validation confidence: 0.8907
Model saved: model_epoch_9.pth

Epoch 10/10


Average train loss: 0.2236


Training precision: 0.6636, Training recall: 0.9968, Average training confidence: 0.9465


Average validation loss: 0.6243


Validation precision: 0.5604, Validation recall: 0.9807, Average validation confidence: 0.9192
Model saved: model_epoch_10.pth


In [ ]:
plot_metrics(train_losses, val_losses, train_precisions, train_recalls, train_confidences, val_precisions, val_recalls, val_confidences, num_epochs)

## Testing

In [ ]:
# Load the Faster R-CNN model with ResNet50 and FPN
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
num_classes = 12
class_names = extract_categories_from_coco_json(test_path)

# Modify the number of output classes
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

# Load the saved weights
model_path = "/models/FasterRCNN/ResNet50_best_model.pth"
model.load_state_dict(torch.load(model_path))
model.eval()

# Configure the device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [ ]:
def visualize_predictions(image, boxes, labels, scores, class_names, threshold=0.35):
    """
    Visualize predictions (bounding boxes and labels) on a single image.
    """
    # Create a color map for the classes
    unique_labels = np.unique(labels)
    colors = {label: tuple(np.random.rand(3)) for label in unique_labels}

    fig, ax = plt.subplots(1, figsize=(image.shape[1] / 30, image.shape[0] / 30))
    ax.imshow(image)
    ax.axis('off')  # Hide the axes

    for box, label, score in zip(boxes, labels, scores):
        if score >= threshold:
            color = colors[label]  # Color for the class
            # Rectangle for the bounding box
            rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                      linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)

            # Label with class and score
            ax.text(box[0], box[1] - 2, f'{class_names[label]}: {score:.2f}',
                    color='black', fontsize=9, fontweight='bold',
                    bbox=dict(facecolor=color, alpha=0.5, edgecolor='none'))

    plt.tight_layout()
    plt.show()
    plt.close(fig)  # Close the figure to save memory

def test_model(model, test_loader, device, class_names, num_classes=12, num_visualizations=8):
    """
    Function for testing the Faster R-CNN model.
    """
    model.to(device)
    model.eval()
    predictions = []
    all_preds = []  # For mAP calculation
    all_targets = []  # For mAP calculation

    print("\nStarting testing...")
    visualizations_done = 0  # Counter for visualizations
    visualized_indices = set()  # Set to track visualized images

    with torch.no_grad():
        for idx, (images, targets) in enumerate(test_loader):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]  # Targets on device

            # Predictions from the model
            preds = model(images)

            # Generate a list of random indices of images to visualize
            available_indices = list(range(len(images)))
            random.shuffle(available_indices)

            for i, (image, pred, target) in enumerate(zip(images, preds, targets)):
                # Add the prediction to the list
                predictions.append({
                    'boxes': pred['boxes'].cpu().numpy(),
                    'labels': pred['labels'].cpu().numpy(),
                    'scores': pred['scores'].cpu().numpy()
                })

                # Collect predictions and targets for mAP calculation
                all_preds.append(pred)
                all_targets.append(target)

                # Select a random index that hasn't been visualized
                random_idx = available_indices.pop()
                if random_idx not in visualized_indices:
                    # Visualize the image
                    visualize_predictions(
                        image.cpu().numpy().transpose(1, 2, 0),  # Convert (C, H, W) -> (H, W, C)
                        pred['boxes'].cpu().numpy(),
                        pred['labels'].cpu().numpy(),
                        pred['scores'].cpu().numpy(),
                        class_names
                    )
                    visualized_indices.add(random_idx)  # Add the index to the set of visualized images
                    visualizations_done += 1

                if visualizations_done >= num_visualizations:
                    break  # Stop when the desired number of visualizations is reached

            if visualizations_done >= num_visualizations:
                break  # Stop the entire loop if we've reached the number of visualizations

    print("Testing finished.")
    return predictions

In [ ]:
predictions = test_model(model, test_loader, device, class_names=extract_categories_from_coco_json(new_coco_json_path), num_classes=num_classes)

In [ ]:
def plot_gt_label_distribution(test_loader, device, class_names, num_classes=12):
    """
    Visualize the distribution of ground truth labels in the test dataset.

    :param test_loader: DataLoader with test data
    :param device: Device to perform computation on (CPU or GPU)
    :param class_names: List of class names
    :param num_classes: Number of classes (including background)
    """
    # Create an array to keep track of GT label frequencies
    gt_label_counts = torch.zeros(num_classes)

    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc="Calculating GT label distribution", leave=False):
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            for target in targets:
                gt_labels = target['labels'].cpu()  # Extract GT labels
                for label in gt_labels:
                    gt_label_counts[label] += 1  # Increment the frequency for the class

    # Convert the tensor to a numpy array to avoid the warning
    gt_label_counts = gt_label_counts.numpy()

    # Convert the list of class names to a numpy array
    class_names_array = np.array(class_names)

    # Plot the distribution
    plt.figure(figsize=(10, 6))
    sns.barplot(x=class_names_array, y=gt_label_counts, palette="viridis")
    plt.title('Distribution of Ground Truth Labels', fontsize=16)
    plt.xlabel('Classes', fontsize=14)
    plt.ylabel('Frequency', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_gt_label_distribution(test_loader, device, class_names)

In [ ]:
def plot_confusion_matrix(conf_matrix, class_names, cmap='Blues'):
    """
    Plot the confusion matrix with a heatmap.

    :param conf_matrix: confusion matrix
    :param class_names: list of class names (including 'background' if necessary)
    :param cmap: color map for the heatmap
    """
    plt.figure(figsize=(10, 8))  # Set figure size
    ax = sns.heatmap(conf_matrix, annot=True, fmt='d', cmap=cmap,
                     xticklabels=class_names, yticklabels=class_names,
                     cbar=False, annot_kws={'size': 12}, square=True)

    plt.title('Confusion Matrix', fontsize=16)
    plt.xlabel('Predicted', fontsize=14)
    plt.ylabel('True', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

def plot_normalized_confusion_matrix(conf_matrix, class_names, cmap='Blues'):
    """Plot the normalized confusion matrix."""
    # Normalize the confusion matrix: each element in a row is divided by the sum of the row
    conf_matrix_normalized = conf_matrix.astype('float')  # Convert to float for division
    row_sums = conf_matrix_normalized.sum(axis=1, keepdims=True)  # Sum for each row
    conf_matrix_normalized /= row_sums  # Divide each row by its sum

    # Visualize the normalized matrix as a heatmap
    plt.figure(figsize=(10, 8))
    ax = sns.heatmap(conf_matrix_normalized, annot=True, fmt='.2f', cmap=cmap,
                     xticklabels=class_names, yticklabels=class_names,
                     cbar_kws={'label': 'Percentage'}, annot_kws={'size': 12}, square=True)

    plt.title('Normalized Confusion Matrix', fontsize=16)
    plt.xlabel('Predicted', fontsize=14)
    plt.ylabel('True', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

def evaluate_test_set_iou(model, test_loader, device, num_classes=12, iou_threshold=0.3, score_threshold=0.3):
    model.eval()
    all_true = []
    all_pred = []

    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc="Testing", leave=False):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            outputs = model(images)

            for target, output in zip(targets, outputs):
                gt_boxes = target['boxes'].cpu()
                gt_labels = target['labels'].cpu()
                pred_boxes = output['boxes'].cpu()
                pred_labels = output['labels'].cpu()
                pred_scores = output['scores'].cpu()

                high_score_idx = pred_scores > score_threshold
                pred_boxes = pred_boxes[high_score_idx]
                pred_labels = pred_labels[high_score_idx]

                ious = box_iou(pred_boxes, gt_boxes)
                for i_pred, box_pred_lbl in enumerate(pred_labels):
                    iou_vals = ious[i_pred]
                    max_iou_idx = torch.argmax(iou_vals)
                    if iou_vals[max_iou_idx] >= iou_threshold:
                        all_true.append(gt_labels[max_iou_idx].item())
                        all_pred.append(box_pred_lbl.item())
                        ious[:, max_iou_idx] = 0
                    else:
                        # No match => class background 0
                        all_true.append(0)
                        all_pred.append(box_pred_lbl.item())

                # For GTs that didn't have any match
                matched_gt = (ious.sum(dim=0) == 0).nonzero().flatten()
                for idx in matched_gt:
                    all_true.append(gt_labels[idx].item())
                    # No prediction => considered background
                    all_pred.append(0)

    cm = confusion_matrix(all_true, all_pred, labels=list(range(num_classes)))
    return cm

In [ ]:
# Use after training:
test_cm = evaluate_test_set_iou(model, test_loader, device, num_classes)

# Visualize the confusion matrix
plot_confusion_matrix(test_cm, class_names)
plot_normalized_confusion_matrix(test_cm, class_names)